# ¿El estilo de juego persiste entre temporadas?

El notebook anterior (`ml_style_clusters.ipynb`) terminó con dos hallazgos y una
deuda. Los hallazgos: el eje que domina el agrupamiento de equipos europeos no es
*cómo* juegan sino *qué tan buenos son* (`r = +0.93` con los puntos por partido),
y no hay estructura de clusters sino un continuo. La deuda: todo eso se midió
sobre **una sola temporada**, y con una foto no se puede distinguir un rasgo
estable de un club de una casualidad de doce meses.

Ahora hay cinco temporadas (2021-22 a 2025-26): **484 equipos-temporada, 130
clubes, 331 pares consecutivos**. Con eso se puede hacer la pregunta que de
verdad valida o tumba todo el ejercicio:

> Si un club juega de cierta manera este año, ¿juega parecido el que viene?

Porque si la respuesta es no, entonces "estilo" no es una propiedad del club —
es ruido anual con nombre bonito, y hablar de arquetipos no tiene sentido.

> **Cómo terminó esto** (conviene saberlo antes de empezar): la respuesta es
> **sí y no, y la distinción importa**. Las *métricas* de estilo persisten de
> verdad — faltas cometidas y fuera de juego provocados al rival correlacionan
> `0.55` consigo mismas de un año al siguiente, incluso después de quitar el
> nivel del equipo. Pero los *grupos* de estilo no: retienen 42% de un año al
> otro, caen a nivel de azar en cuatro años, y **ninguno de los seis sobrevive
> a un bootstrap**. O sea: el estilo existe y se mide, pero como **ejes
> continuos**, no como tribus. Los arquetipos hay que jubilarlos.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from viz_theme import (CLUSTER_COLORS, SEQUENTIAL_BLUE, DIVERGING, INK,
                       apply_plotly_theme, render_plot)

apply_plotly_theme(opaque_surface=True)
pd.set_option('display.max_columns', 40)
RNG = 42
rng = np.random.default_rng(0)

SEASONS = ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

## 1. Los datos: un panel, no una foto

`teams_all_seasons.csv` lo arma `consolidate_data.py` a partir de lo que bajan
`fetch_fbref.py` (equipos) y `fetch_understat.py` (jugadores). Trae dos cosas que
el notebook anterior no tenía:

1. **Cinco temporadas** en vez de una.
2. **Las tablas "vs equipo"** — lo que los rivales le hacen a cada equipo. Son
   la mitad que faltaba del estilo: no solo cuántos centros tiras, también
   cuántos te tiran; no solo cuántas faltas cometes, también cuántas provocas.

In [2]:
df = pd.read_csv('../data/processed/teams_all_seasons.csv', encoding='utf-8-sig')

# tasas por 90', propias y recibidas
per90 = {}
for stat in ['Crs', 'Off', 'Fls', 'Fld', 'CrdY', 'Int', 'TklW']:
    per90[stat] = df[f'ms_Performance_{stat}'] / df['ms_90s']
    per90[f'vs_{stat}'] = df[f'vs_ms_Performance_{stat}'] / df['ms_90s']
per90['Sh'] = df['sh_Standard_Sh/90']
per90['vs_Sh'] = df['vs_sh_Standard_Sh/90']
per90['Poss'] = df['ov_Poss']
df = pd.concat([df, pd.DataFrame(per90)], axis=1)

print(f'{len(df)} equipos-temporada · {df.Squad.nunique()} clubes distintos')
print('\nClubes por nº de temporadas en que aparecen:')
print(df.groupby('Squad')['temporada'].nunique().value_counts().sort_index(ascending=False).to_string())

484 equipos-temporada · 130 clubes distintos

Clubes por nº de temporadas en que aparecen:
temporada
5    64
4    19
3    13
2    15
1    19


In [3]:
# pares (t, t+1): la unidad de análisis de todo el notebook
pares_por_salto = []
for s1, s2 in zip(SEASONS, SEASONS[1:]):
    comunes = set(df[df.temporada == s1].Squad) & set(df[df.temporada == s2].Squad)
    pares_por_salto.append((f'{s1} -> {s2}', len(comunes)))
    print(f'{s1} -> {s2}: {len(comunes)} clubes en ambas')
print(f'\nTOTAL: {sum(n for _, n in pares_por_salto)} pares consecutivos')
print(f'Clubes que cambiaron de liga en 5 años: {(df.groupby("Squad")["liga"].nunique() > 1).sum()}')

2021-22 -> 2022-23: 84 clubes en ambas
2022-23 -> 2023-24: 83 clubes en ambas
2023-24 -> 2024-25: 82 clubes en ambas
2024-25 -> 2025-26: 82 clubes en ambas

TOTAL: 331 pares consecutivos
Clubes que cambiaron de liga en 5 años: 0


**Ningún club cambió de liga.** Los que descienden simplemente salen del dataset
(dejan de estar en el top 5) y los que ascienden aparecen. Eso simplifica la
identidad del club, pero crea el problema del que se ocupa la sección siguiente:
la liga queda **perfectamente confundida** con el club.

### Selección de features

Se separan en dos grupos a propósito, porque el notebook anterior mostró que se
comportan distinto:

- **Conducta** (14): centros, fuera de juego, faltas cometidas y recibidas,
  amarillas, intercepciones y tackles — cada una en versión propia y recibida.
  Describen *qué hace* el equipo y *qué le hacen*.
- **Volumen** (3): posesión, tiros y tiros concedidos. Se incluyen para
  comparar, marcadas como contaminadas por nivel: ya sabemos que correlacionan
  ~0.78 con los puntos.

In [4]:
CONDUCTA = {
    'Fls': 'Faltas cometidas', 'Fld': 'Faltas recibidas', 'CrdY': 'Amarillas',
    'Crs': 'Centros', 'Off': 'Fuera de juego', 'Int': 'Intercepciones',
    'TklW': 'Tackles ganados',
    'vs_Fls': 'Faltas que provoca el rival', 'vs_Fld': 'Faltas recibidas del rival',
    'vs_CrdY': 'Amarillas del rival', 'vs_Crs': 'Centros recibidos',
    'vs_Off': 'Fuera de juego del rival', 'vs_Int': 'Intercepciones del rival',
    'vs_TklW': 'Tackles del rival',
}
VOLUMEN = {'Poss': 'Posesión', 'Sh': 'Tiros', 'vs_Sh': 'Tiros concedidos'}
TODAS = {**CONDUCTA, **VOLUMEN}
cols = list(TODAS)
print(f'{len(CONDUCTA)} de conducta + {len(VOLUMEN)} de volumen = {len(cols)} métricas')

14 de conducta + 3 de volumen = 17 métricas


## 2. Normalización: qué se le quita a los datos y qué no

Hay una decisión acá que cambia el resultado, así que va argumentada con números
y no por costumbre.

**Lo que sí hay que quitar: la media de liga-temporada.** Por dos razones
medibles. Primero, hay *drift*: las faltas por partido cayeron de 449 a 429 y las
amarillas de 78 a 71 entre 2021-22 y 2025-26 — cambio de criterio arbitral, no de
estilo. Segundo, y más grave: **como ningún club cambió de liga, la identidad de
la liga está perfectamente confundida con la del club**. La Liga saca 2.40
amarillas por partido y Ligue 1 1.88; sin quitar eso, parte de lo que llamaríamos
"su estilo persiste" sería en realidad "el club sigue jugando en España".

**Lo que NO hay que hacer: dividir por la desviación de cada liga-temporada.**
Se probó y da lo mismo (ICC de amarillas 0.43 con y sin), pero borra las
diferencias reales de dispersión entre años y ligas. Así que se centra por
liga-temporada y se escala con **una sola desviación global por métrica**, que
deja las métricas comparables entre sí sin destruir esa información.

> **Un miedo razonable que conviene despejar**: ¿el z-score no aplasta a los
> outliers, que en fútbol suelen ser lo interesante? No. Es una transformación
> lineal: Barcelona con 68.9% de posesión queda en +2.88 desviaciones, Bayern en
> +2.46, y la razón de distancias al centro pasa de 1.92 a 1.91. Lo que sí
> destruye outliers es winsorizar, recortar, o pasar a rangos/percentiles — nada
> de eso se hace acá. Y en este análisis los outliers no son un estorbo: son la
> señal. Si Bayern está en +2.5 los cinco años, eso *es* la persistencia.

In [5]:
N = df[['temporada', 'liga', 'Squad']].copy()
for c in cols:
    centrado = df[c] - df.groupby(['liga', 'temporada'])[c].transform('mean')
    N[c] = centrado / centrado.std()

# el nivel, centrado igual, para poder descontarlo después
N['ppm'] = (df['pt_Team Success_PPM']
            - df.groupby(['liga', 'temporada'])['pt_Team Success_PPM'].transform('mean'))

# residuo: la misma métrica con el nivel quitado por regresión
R = N[['temporada', 'liga', 'Squad']].copy()
for c in cols:
    R[c] = N[c] - np.polyval(np.polyfit(N['ppm'], N[c], 1), N['ppm'])

print('Verificación — media por liga-temporada tras centrar (debe ser ~0):')
print(f"  máximo absoluto: {N.groupby(['liga','temporada'])[cols].mean().abs().max().max():.1e}")

Verificación — media por liga-temporada tras centrar (debe ser ~0):
  máximo absoluto: 1.6e-15


## 3. ¿Persiste cada métrica?

Dos medidas complementarias, sobre los 331 pares:

- **`r(t, t+1)`** — correlación entre el valor de un club en una temporada y el
  de la siguiente. Directa e interpretable.
- **ICC** — qué fracción de toda la variación vive **entre** clubes y no
  **dentro** de un mismo club a lo largo de los años. Es la respuesta de libro a
  "¿esto pertenece al club o al año?", y maneja bien que unos clubes tengan 5
  temporadas y otros 2.

Y las dos se calculan también sobre el **residuo sin nivel**. Esa es la columna
que decide: si una métrica persiste igual con y sin el descuento, es estilo; si
se derrumba, era el nivel del equipo disfrazado.

Umbrales declarados **antes** de mirar: ICC > 0.5 = rasgo de club · 0.2-0.5 =
tendencia débil · < 0.2 = es la temporada, no el club.

In [6]:
def pares_consecutivos(frame, col):
    a, b = [], []
    for s1, s2 in zip(SEASONS, SEASONS[1:]):
        m = (frame[frame.temporada == s1][['Squad', col]]
             .merge(frame[frame.temporada == s2][['Squad', col]], on='Squad',
                    suffixes=('_t', '_t1')))
        a.append(m[col + '_t'].values); b.append(m[col + '_t1'].values)
    return np.concatenate(a), np.concatenate(b)


def icc(frame, col):
    '''ICC(1) para panel desbalanceado: varianza entre clubes / varianza total.'''
    d = frame[['Squad', col]].dropna()
    g = d.groupby('Squad')[col]
    k, m = g.size(), g.mean()
    n, ng, glob = len(d), g.ngroups, d[col].mean()
    msb = (k * (m - glob) ** 2).sum() / (ng - 1)
    msw = g.apply(lambda x: ((x - x.mean()) ** 2).sum()).sum() / (n - ng)
    k0 = (n - (k ** 2).sum() / n) / (ng - 1)   # tamaño de grupo efectivo
    return (msb - msw) / (msb + (k0 - 1) * msw)


filas = []
for c in cols:
    a, b = pares_consecutivos(N, c)
    ar, br = pares_consecutivos(R, c)
    filas.append({
        'Métrica': TODAS[c], 'Tipo': 'conducta' if c in CONDUCTA else 'volumen',
        'r(t,t+1)': np.corrcoef(a, b)[0, 1],
        'r sin nivel': np.corrcoef(ar, br)[0, 1],
        'ICC': icc(N, c), 'ICC sin nivel': icc(R, c),
        'corr con PPM': np.corrcoef(N[c], N['ppm'])[0, 1],
    })
tabla = pd.DataFrame(filas).sort_values('r sin nivel', ascending=False).reset_index(drop=True)
tabla.round(2)

,Métrica,Tipo,"r(t,t+1)",r sin nivel,ICC,ICC sin nivel,corr con PPM
0,Fuera de juego del rival,conducta,0.55,0.55,0.38,0.38,0.03
1,Faltas cometidas,conducta,0.67,0.55,0.54,0.41,-0.43
2,Faltas recibidas del rival,conducta,0.66,0.54,0.53,0.41,-0.43
3,Tackles del rival,conducta,0.55,0.54,0.36,0.34,0.13
4,Posesión,volumen,0.84,0.50,0.78,0.42,0.77
5,Tackles ganados,conducta,0.54,0.50,0.39,0.36,-0.24
6,Intercepciones del rival,conducta,0.62,0.47,0.52,0.32,0.51
7,Centros,conducta,0.47,0.46,0.37,0.36,0.11
8,Faltas recibidas,conducta,0.46,0.45,0.34,0.34,-0.06
9,Faltas que provoca el rival,conducta,0.45,0.44,0.34,0.33,-0.06


In [7]:
cond = tabla[tabla.Tipo == 'conducta']
vol = tabla[tabla.Tipo == 'volumen']

fig = go.Figure()
for sub, nombre, color in [(cond, 'Conducta', SEQUENTIAL_BLUE[5]), (vol, 'Volumen', DIVERGING['neg'])]:
    fig.add_trace(go.Scatter(
        x=sub['corr con PPM'].abs(), y=sub['r sin nivel'], mode='markers+text',
        name=nombre, text=sub['Métrica'], textposition='middle right',
        textfont=dict(size=9.5, color=INK['secondary']),
        marker=dict(size=11, color=color, opacity=0.9,
                    line=dict(color=INK['surface'], width=1.5)),
        hovertemplate='<b>%{text}</b><br>persistencia sin nivel: %{y:.2f}'
                      '<br>|corr| con puntos: %{x:.2f}<extra></extra>',
    ))
fig.add_vline(x=0.30, line=dict(color=INK['axis'], width=1, dash='dash'))
fig.add_hline(y=0.45, line=dict(color=INK['axis'], width=1, dash='dash'))
fig.add_annotation(x=0.02, y=0.60, text='<b>estilo puro</b><br>persiste y no mide nivel',
                   showarrow=False, xanchor='left', align='left',
                   font=dict(size=10.5, color=INK['muted']))
fig.update_layout(
    title=dict(text='Qué métricas son estilo de verdad',
               subtitle=dict(text='Arriba a la izquierda: se repiten año a año <b>y</b> no dicen '
                                  'nada de qué tan bueno es el equipo')),
    xaxis=dict(title='|correlación con los puntos por partido|  (más a la derecha = más nivel)',
               range=[-0.03, 0.92]),
    yaxis=dict(title='Persistencia t → t+1, con el nivel descontado'),
    legend=dict(orientation='h', y=-0.16, x=0),
)
render_plot(fig, width=840, height=620)

### El resultado principal

**Sí persiste, y bastante.** Pero lo interesante es *cuáles*:

- **Las de volumen persisten más que nada... hasta que se les quita el nivel.**
  Posesión pasa de `0.84` a `0.50`, y Tiros se desploma de `0.73` a `0.27`. Lo
  que se repetía año a año no era la forma de jugar: era que los equipos buenos
  siguen siendo buenos. Es la misma trampa de la sección 5.1 del notebook
  anterior, ahora vista en el tiempo.
- **Las de conducta casi no se mueven al descontar el nivel.** *Fuera de juego
  del rival* pasa de `0.55` a `0.55` — cero pérdida, y su correlación con los
  puntos es `0.03`. *Tackles del rival* `0.55 → 0.54`, *Centros* `0.47 → 0.46`,
  *Faltas recibidas* `0.46 → 0.45`.
- **El estilo más persistente es la aspereza.** *Faltas cometidas* (`0.55` sin
  nivel, ICC 0.41) y su espejo *faltas que provoca el rival* (`0.54`) son las
  métricas de conducta más estables del estudio. Cómo interrumpe un equipo es
  su rasgo más duradero.

Y algo que justifica el trabajo de scraping de esta sesión: **tres de las cinco
métricas de estilo más limpias son de las tablas "vs equipo"** — fuera de juego
del rival, tackles del rival, faltas que provoca. Datos que no existían en el
proyecto hasta ahora.

In [8]:
orden = tabla.sort_values('ICC sin nivel')
fig = go.Figure()
fig.add_trace(go.Bar(
    y=orden['Métrica'], x=orden['ICC'], name='ICC (crudo)', orientation='h',
    marker=dict(color=SEQUENTIAL_BLUE[1], line=dict(width=0)),
    hovertemplate='<b>%{y}</b><br>ICC crudo: %{x:.2f}<extra></extra>',
))
fig.add_trace(go.Bar(
    y=orden['Métrica'], x=orden['ICC sin nivel'], name='ICC con el nivel descontado',
    orientation='h', marker=dict(color=SEQUENTIAL_BLUE[6], line=dict(width=0)),
    hovertemplate='<b>%{y}</b><br>ICC sin nivel: %{x:.2f}<extra></extra>',
))
for x, t in [(0.5, 'rasgo de club'), (0.2, 'es el año')]:
    fig.add_vline(x=x, line=dict(color=INK['axis'], width=1, dash='dash'))
    fig.add_annotation(x=x, y=-0.8, text=t, showarrow=False,
                       font=dict(size=9.5, color=INK['muted']), yanchor='top')
fig.update_layout(
    barmode='group', bargap=0.25,
    title=dict(text='¿Cuánto de cada métrica pertenece al club y no al año?',
               subtitle=dict(text='ICC: fracción de la variación que vive entre clubes · '
                                  'la barra oscura es la que cuenta')),
    xaxis=dict(title='ICC'), yaxis=dict(title=None),
    legend=dict(orientation='h', y=-0.13, x=0), margin=dict(l=200, b=90),
)
render_plot(fig, width=820, height=640)

Con los umbrales declarados antes: **ninguna métrica de conducta llega a 0.5**
una vez descontado el nivel — las mejores (faltas cometidas y faltas provocadas,
0.41) se quedan en "tendencia fuerte pero no determinante". Todas las de conducta
superan 0.2, así que ninguna es puro ruido anual.

Traducido: **un club tiene una forma de jugar reconocible, pero explica menos de
la mitad de lo que hace en una temporada dada.** El resto es el año — plantilla,
entrenador, lesiones, calendario.

## 4. ¿O es solo cosa de los extremos?

Una sospecha razonable: quizás la persistencia no es un fenómeno general sino de
los casos extremos. Bayern siempre es Bayern y Getafe siempre es Getafe, pero en
el montón del medio no hay estilo estable, solo ruido.

Se comprueba recalculando el ICC **quitando el 10% de cada cola** de cada
métrica. Si la persistencia se desploma, la sospecha era correcta.

> **Pero hay una trampa, y es fatal si no se controla**: recortar el rango baja
> la correlación **por construcción**, aunque la persistencia sea perfectamente
> uniforme. Cualquier caída que se observe tiene una parte de artefacto. Así que
> hay que comparar contra un nulo: datos simulados con **la misma estructura de
> varianza** (mismo nº de temporadas por club, misma partición entre/dentro) pero
> gaussianos — es decir, sin ninguna razón para que los extremos sean más
> estables. Se recortan igual y se mide cuánto cae el ICC solo por el recorte.

In [9]:
def icc_recortado(frame, col, q=0.10):
    lo, hi = frame[col].quantile([q, 1 - q])
    medio = frame[(frame[col] > lo) & (frame[col] < hi)]
    return icc(medio[medio.groupby('Squad')['Squad'].transform('size') >= 2], col)


def caida_nula(frame, col, n_sim=40):
    '''Cuánto cae el ICC al recortar, en un panel donde la persistencia es
    uniforme por construcción.'''
    d = frame[['Squad', col]].dropna()
    g = d.groupby('Squad')[col]
    tam = g.size()
    var_dentro = g.apply(lambda x: x.var(ddof=1) if len(x) > 1 else np.nan).mean()
    var_entre = max(g.mean().var(ddof=1) - var_dentro / tam.mean(), 1e-9)
    caidas = []
    for _ in range(n_sim):
        # un efecto fijo por club (eso es lo que crea la persistencia) más
        # ruido independiente por temporada
        efecto = {s: rng.normal(0, np.sqrt(var_entre)) for s in tam.index}
        sim = pd.DataFrame(
            [(s, efecto[s] + rng.normal(0, np.sqrt(var_dentro)))
             for s, k in tam.items() for _ in range(k)],
            columns=['Squad', col])
        caidas.append(icc(sim, col) - icc_recortado(sim, col))
    return np.mean(caidas), np.std(caidas)


filas = []
for c in cols:
    real = icc(N, c) - icc_recortado(N, c)
    mu, sd = caida_nula(N, c)
    filas.append({'Métrica': TODAS[c], 'Caída real': real,
                  'Caída esperada por el recorte': mu, 'Exceso': real - mu,
                  'z': (real - mu) / sd if sd > 0 else np.nan})
t_rec = pd.DataFrame(filas).sort_values('Exceso', ascending=False).reset_index(drop=True)
print(f"Métricas donde la caída supera al nulo con z > 2: {(t_rec['z'] > 2).sum()} de {len(t_rec)}")
t_rec.round(2)

Métricas donde la caída supera al nulo con z > 2: 0 de 17


,Métrica,Caída real,Caída esperada por el recorte,Exceso,z
0,Faltas recibidas,0.24,0.16,0.08,1.55
1,Amarillas,0.24,0.18,0.05,1.14
2,Centros,0.23,0.18,0.04,0.78
3,Tiros concedidos,0.23,0.20,0.03,0.71
4,Centros recibidos,0.23,0.21,0.03,0.70
5,Faltas recibidas del rival,0.23,0.21,0.02,0.40
6,Faltas que provoca el rival,0.19,0.17,0.01,0.27
7,Tiros,0.19,0.19,0.01,0.13
8,Faltas cometidas,0.19,0.20,-0.01,-0.18
9,Fuera de juego,0.12,0.13,-0.01,-0.21


In [10]:
o = t_rec.sort_values('Caída real')
fig = go.Figure()
fig.add_trace(go.Bar(y=o['Métrica'], x=o['Caída real'], name='Caída observada',
                     orientation='h', marker=dict(color=SEQUENTIAL_BLUE[6], line=dict(width=0)),
                     hovertemplate='<b>%{y}</b><br>cae %{x:.2f}<extra></extra>'))
fig.add_trace(go.Scatter(y=o['Métrica'], x=o['Caída esperada por el recorte'],
                         name='Lo que cae solo por recortar (nulo)', mode='markers',
                         marker=dict(size=10, color=DIVERGING['neg'], symbol='diamond',
                                     line=dict(color=INK['surface'], width=1.5)),
                         hovertemplate='<b>%{y}</b><br>nulo: %{x:.2f}<extra></extra>'))
fig.update_layout(
    title=dict(text='La caída al quitar los extremos es artefacto, no hallazgo',
               subtitle=dict(text='Si el rombo cae sobre la barra, todo lo observado se explica '
                                  'por el recorte del rango')),
    xaxis=dict(title='Caída del ICC'), yaxis=dict(title=None),
    legend=dict(orientation='h', y=-0.13, x=0), margin=dict(l=200, b=80),
)
render_plot(fig, width=820, height=620)

### La sospecha no se sostiene

**Cero de 17 métricas** tienen una caída que supere lo que produce el recorte por
sí solo (ninguna llega a z > 2). Las caídas observadas —entre 0.12 y 0.24— son
casi exactamente las que predice el nulo, entre 0.13 y 0.22.

O sea: **la persistencia no está concentrada en los extremos.** Un equipo del
montón repite su forma de jugar tanto como Bayern repite la suya; lo que pasa es
que su forma de jugar es menos distinguible del promedio, que es otra cosa.

Vale la pena subrayar por qué hacía falta el nulo: sin él, la tabla de caídas
(0.24 en faltas recibidas, 0.24 en amarillas, 0.23 en centros) parecía una
confirmación limpia de la sospecha. El control la desarma entera.

## 5. ¿Y los grupos de estilo?

Hasta acá se midió métrica por métrica. Pero el notebook anterior no hablaba de
métricas sino de **arquetipos**: seis grupos con nombre. La pregunta natural es
si un club se queda en su grupo.

Se ajusta el clustering **una sola vez sobre los 484 equipos-temporada** — no uno
por temporada — para que las etiquetas sean directamente comparables sin tener
que alinear nada. Después se mide qué fracción de los 331 pares conserva el
grupo, contra lo que daría el azar con esos tamaños de cluster.

In [11]:
X = N[list(CONDUCTA)].values
var_acum = np.cumsum(PCA(random_state=RNG).fit(X).explained_variance_ratio_)
n_comp = int(np.searchsorted(var_acum, 0.80) + 1)
scores = PCA(n_components=n_comp, whiten=True, random_state=RNG).fit_transform(X)

K = 6
N['grupo'] = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit_predict(scores)
print(f'{len(CONDUCTA)} features de conducta -> {n_comp} componentes ({var_acum[n_comp-1]:.0%}), blanqueados')
print('Tamaño de cada grupo:', N.grupo.value_counts().sort_index().to_dict())

iguales = total = 0
for s1, s2 in zip(SEASONS, SEASONS[1:]):
    m = (N[N.temporada == s1][['Squad', 'grupo']]
         .merge(N[N.temporada == s2][['Squad', 'grupo']], on='Squad', suffixes=('_t', '_t1')))
    iguales += (m.grupo_t == m.grupo_t1).sum(); total += len(m)

azar = (N.grupo.value_counts(normalize=True) ** 2).sum()
print(f'\nMismo grupo en t y t+1 : {iguales}/{total} = {iguales/total:.1%}')
print(f'Esperado por azar      : {azar:.1%}')
print(f"Cohen's kappa          : {(iguales/total - azar)/(1 - azar):.2f}  (0 = azar, 1 = perfecto)")

14 features de conducta -> 7 componentes (83%), blanqueados
Tamaño de cada grupo: {0: 85, 1: 74, 2: 100, 3: 63, 4: 97, 5: 65}

Mismo grupo en t y t+1 : 140/331 = 42.3%
Esperado por azar      : 17.2%
Cohen's kappa          : 0.30  (0 = azar, 1 = perfecto)


In [12]:
# retención según cuántas temporadas de distancia haya
puntos = []
for salto in range(1, 5):
    ig = tot = 0
    for i in range(len(SEASONS) - salto):
        m = (N[N.temporada == SEASONS[i]][['Squad', 'grupo']]
             .merge(N[N.temporada == SEASONS[i + salto]][['Squad', 'grupo']],
                    on='Squad', suffixes=('_a', '_b')))
        ig += (m.grupo_a == m.grupo_b).sum(); tot += len(m)
    puntos.append((salto, ig / tot, tot))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[p[0] for p in puntos], y=[p[1] * 100 for p in puntos], mode='lines+markers',
    name='Retención observada', line=dict(color=SEQUENTIAL_BLUE[6], width=2.5),
    marker=dict(size=11, line=dict(color=INK['surface'], width=2)),
    customdata=[p[2] for p in puntos],
    hovertemplate='A %{x} temporada(s) de distancia<br>%{y:.0f}%% conserva el grupo'
                  '<br>(%{customdata} pares)<extra></extra>',
))
fig.add_hline(y=azar * 100, line=dict(color=DIVERGING['neg'], width=2, dash='dash'))
fig.add_annotation(x=4, y=azar * 100, text='azar', showarrow=False, xanchor='right',
                   yanchor='bottom', font=dict(size=10.5, color=DIVERGING['neg']))
fig.update_layout(
    title=dict(text='El grupo de estilo se olvida en cuatro años',
               subtitle=dict(text='Fracción de clubes que siguen en el mismo grupo, según '
                                  'cuántas temporadas pasaron')),
    xaxis=dict(title='Temporadas de distancia', dtick=1),
    yaxis=dict(title='Conservan el grupo (%)', range=[0, 50]),
    showlegend=False,
)
render_plot(fig, width=760, height=460)

**42.3% de retención contra 17.2% de azar** (kappa 0.30). Es real —dos veces y
media el azar— pero es una concordancia modesta: de cada tres clubes, dos cambian
de grupo de un año al otro.

Y **a cuatro temporadas de distancia cae a 20.8%, prácticamente el azar.** El
grupo de estilo de un club en 2021-22 no dice casi nada de dónde está en 2025-26.

## 6. El bootstrap: ¿existen esos grupos siquiera?

Antes de interpretar el 42% conviene una pregunta más básica, que es la deuda
explícita que dejó el notebook anterior: **¿los grupos son estables ante un
cambio en los datos de entrada?**

Se remuestrean **clubes** (no equipos-temporada — si no, un mismo club aparecería
partido entre la muestra y la que queda fuera, y eso infla la estabilidad), se
reajusta todo el pipeline, y para cada grupo original se busca el grupo más
parecido en el nuevo ajuste. El criterio estándar es el de Hennig: Jaccard medio
por encima de 0.75 = cluster real; entre 0.60 y 0.75 = patrón; por debajo de
0.60 = se deshace.

In [13]:
clubes = N.Squad.unique()
jac = {k: [] for k in range(K)}

for _ in range(60):
    sel = np.unique(rng.choice(clubes, size=len(clubes), replace=True))
    idx = np.flatnonzero(N.Squad.isin(sel).values)
    sb = PCA(n_components=n_comp, whiten=True, random_state=RNG).fit_transform(X[idx])
    lb = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit_predict(sb)
    orig = N.grupo.values[idx]
    for k in range(K):
        A = set(np.flatnonzero(orig == k))
        if A:
            jac[k].append(max(len(A & set(np.flatnonzero(lb == j))) /
                              len(A | set(np.flatnonzero(lb == j))) for j in range(K)))

boot = pd.DataFrame({
    'Grupo': range(K),
    'Equipos-temporada': [int((N.grupo == k).sum()) for k in range(K)],
    'Jaccard medio': [np.mean(jac[k]) for k in range(K)],
})
boot['Veredicto'] = np.where(boot['Jaccard medio'] > 0.75, 'cluster real',
                     np.where(boot['Jaccard medio'] > 0.60, 'patrón', 'se deshace'))
boot.round(2)

,Grupo,Equipos-temporada,Jaccard medio,Veredicto
0,0,85,0.29,se deshace
1,1,74,0.35,se deshace
2,2,100,0.41,se deshace
3,3,63,0.36,se deshace
4,4,97,0.40,se deshace
5,5,65,0.44,se deshace


In [14]:
fig = go.Figure(go.Bar(
    x=[f'Grupo {k}' for k in boot.Grupo], y=boot['Jaccard medio'],
    marker=dict(color=[CLUSTER_COLORS[k] for k in boot.Grupo], line=dict(width=0)),
    customdata=boot['Equipos-temporada'],
    hovertemplate='<b>%{x}</b><br>Jaccard: %{y:.2f}<br>%{customdata} equipos-temporada<extra></extra>',
))
for y, t, c in [(0.75, 'cluster real', DIVERGING['pos']), (0.60, 'patrón', INK['axis'])]:
    fig.add_hline(y=y, line=dict(color=c, width=1.5, dash='dash'))
    fig.add_annotation(x=5.5, y=y, text=t, showarrow=False, xanchor='right', yanchor='bottom',
                       font=dict(size=10, color=c))
fig.update_layout(
    title=dict(text='Ninguno de los seis grupos sobrevive al bootstrap',
               subtitle=dict(text='Similitud media con el grupo equivalente al reajustar sobre '
                                  'una remuestra de clubes')),
    xaxis=dict(title=None), yaxis=dict(title='Jaccard medio', range=[0, 1]),
    showlegend=False,
)
render_plot(fig, width=760, height=470)

**Los seis grupos se deshacen** (Jaccard entre 0.29 y 0.44, todos por debajo del
0.60 mínimo). Cambiando qué clubes entran a la muestra, los grupos que salen son
otros.

Esto no contradice nada: **confirma en cinco temporadas lo que el test de nulo
del notebook anterior ya había mostrado en una**. No hay tribus con fronteras; hay
una nube continua que k-means parte donde le toca. El 42% de retención de la
sección 5 hay que leerlo con esa luz: no es que los clubes "cambien de arquetipo",
es que los arquetipos no son entidades lo bastante sólidas como para cambiarse.

## Conclusión

**El estilo existe, persiste, y no es lo que estábamos midiendo.**

1. **Las métricas de estilo sí son un rasgo del club.** Después de descontar el
   nivel, faltas cometidas y faltas provocadas al rival correlacionan `0.55`
   consigo mismas de una temporada a la siguiente; fuera de juego del rival,
   `0.55` sin perder nada al descontar (su correlación con los puntos es `0.03`).
   Ninguna métrica de conducta baja de ICC 0.2. Un club tiene una manera de
   jugar, y se le nota año tras año.
2. **Pero explica menos de la mitad.** Ningún ICC de conducta llega a 0.5 con el
   nivel descontado: el mejor es 0.41. La mayor parte de lo que hace un equipo en
   una temporada es de esa temporada, no del club.
3. **Los arquetipos hay que jubilarlos.** Retienen 42% de un año al otro, caen al
   azar en cuatro, y ninguno de los seis sobrevive al bootstrap. La forma correcta
   de hablar de estilo en estos datos es por **ejes continuos** —cuánto interrumpe,
   por dónde ataca, cuánto deja que le lleguen— no por grupos con nombre.
4. **Lo de volumen nunca fue estilo.** Posesión y tiros son las métricas más
   persistentes del estudio en crudo (0.84 y 0.73) y las que más se derrumban al
   descontar el nivel (0.50 y 0.27). Persistían porque los equipos buenos siguen
   siendo buenos.

**Qué habilitó este notebook que no se podía antes**: las cinco temporadas (nada
de esto es medible con una) y las tablas "vs equipo" — tres de las cinco métricas
de estilo más limpias son de ahí.

**La limitación principal, declarada**: no hay datos de cambio de entrenador. Un
club que cambia de DT y de estilo baja la persistencia por una razón perfectamente
legítima, y no se puede separar del ruido. Es el confusor no medido más
importante, y probablemente explique parte de ese techo de 0.41.